##PARTE 2: Impacto Financiero y JOINS (6 Ejercicios de Negocio)


##11.El Costo del Marketing Inútil (Agregación con Filtro): Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida.

In [0]:
%sql
SELECT 
    COUNT(distribution_channel) * 1.5 AS costo_perdido_cancelaciones
FROM cursodataacademy.hotel_db.tb_reservas
WHERE is_canceled = 1 AND distribution_channel = 'TA/TO';

##12.El Costo de Cancelar Tarde (DATE_DIFF): (Considera que tienes las columnas arrival_date y reservation_status_date ambas en formato DATE). Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.

In [0]:
%sql
SELECT
    COUNT(*) AS reservas_canceladas_tarde,
    COUNT(*) * 120 AS perdida_total
FROM cursodataacademy.hotel_db.tb_reservas
WHERE is_canceled = 1
  AND DATEDIFF(arrival_date, reservation_status_date) < 3;

##13.(JOIN 1 - LEFT JOIN - Entendiendo el Modelo): Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais (ej. PRT, ESP) y nombre_pais. Une tu tabla tb_reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.hotel_db.tb_paises AS 
SELECT DISTINCT country AS cod_pais, country AS nombre_pais
FROM cursodataacademy.hotel_db.tb_reservas;

In [0]:
%sql
SELECT
    A.is_canceled,
    B.nombre_pais
FROM cursodataacademy.hotel_db.tb_reservas AS A
LEFT JOIN cursodataacademy.hotel_db.tb_paises AS B
    ON A.country = B.cod_pais;

##14.(JOIN 2 - INNER JOIN con Agrupación): Usando el mismo JOIN del ejercicio anterior, averigua: ¿Cuáles son los 5 países (muestra el nombre_pais) que tienen la mayor cantidad de reservas canceladas?

In [0]:
%sql
SELECT
    B.nombre_pais,
    COUNT(*) AS total_canceladas
FROM cursodataacademy.hotel_db.tb_reservas AS A
INNER JOIN cursodataacademy.hotel_db.tb_paises AS B
    ON A.country = B.cod_pais
WHERE A.is_canceled = 1
GROUP BY B.nombre_pais
ORDER BY total_canceladas DESC
LIMIT 5;

##15.(JOIN 3 - Simulación FULL OUTER JOIN): Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor para identificar si hay reservas sin vendedor o vendedores sin reservas asignadas.

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.hotel_db.tb_vendedores AS
SELECT DISTINCT
    CAST(agent AS STRING) AS id_vendedor,
    CONCAT('Vendedor ', agent) AS nombre_vendedor
FROM cursodataacademy.hotel_db.tb_reservas
WHERE agent IS NOT NULL;

In [0]:
%sql
SELECT
    A.distribution_channel,
    B.nombre_vendedor
FROM cursodataacademy.hotel_db.tb_reservas AS A
FULL OUTER JOIN cursodataacademy.hotel_db.tb_vendedores AS B
    ON CAST(A.agent AS STRING) = B.id_vendedor;

##16.(JOIN 4 - RIGHT JOIN): El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel (hotel).

In [0]:
%sql
SELECT 
    A.hotel,
    B.monto_facturado
FROM cursodataacademy.hotel_db.tb_reservas AS A
RIGHT JOIN cursodataacademy.hotel_db.tb_facturas AS B
  ON CAST(A.id_reserva AS INT) = B.id_reserva;